# 🔊 Environmental Sound Classifier — Live Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sardorbekmuminov/esc50-sound-classifier/blob/main/demo.ipynb)

Run this notebook to try the classifier in your browser — **no training needed**. It downloads the pre-trained model (ResNet18 fine-tuned on ESC-50, ~71% test accuracy) and starts an interactive web app.

**How to use:** *Runtime → Run all*, wait ~1–2 minutes, then open the **public link** printed by the last cell. Upload an audio file or record with your microphone; the app shows the top-5 predictions with confidence scores.

Training code, experiments and results: see `esc50_sound_classifier.ipynb`.

## 1. Install dependencies

In [ ]:
!pip install -q timm librosa gradio soundfile

## 2. Download the trained model
The weights are stored in this repository's **Releases** (too large for a normal commit); the class list is in the repository itself.

In [ ]:
import json, urllib.request
from pathlib import Path

REPO = "sardorbekmuminov/esc50-sound-classifier"
WEIGHTS_URL = f"https://github.com/{REPO}/releases/download/v1.0/fine_tuned_resnet18.pt"
CLASSES_URL = f"https://raw.githubusercontent.com/{REPO}/main/classes.json"

for url, name in [(WEIGHTS_URL, "fine_tuned_resnet18.pt"), (CLASSES_URL, "classes.json")]:
    if Path(name).exists():
        continue
    try:
        urllib.request.urlretrieve(url, name)
        print("Downloaded:", name)
    except Exception as e:
        raise RuntimeError(f"Could not download {name} from {url}\n{e}\n"
                           "Check that the repository is public and the v1.0 release contains the file.")

## 3. Load the model

In [ ]:
import numpy as np, torch, timm, librosa

SR, DURATION = 22050, 5
N_SAMPLES = SR * DURATION
N_FFT, HOP, N_MELS = 1024, 512, 128

CLASSES = json.load(open("classes.json"))
model = timm.create_model("resnet18", pretrained=False, num_classes=len(CLASSES), in_chans=1)
model.load_state_dict(torch.load("fine_tuned_resnet18.pt", map_location="cpu"))
model.eval()
print(f"Model loaded: {len(CLASSES)} classes")

## 4. Inference
Audio → 5-second windows → log-mel spectrogram → ResNet18 → probabilities. Clips longer than 5 s are split into windows and the probabilities are averaged.

In [ ]:
def fix_length(sig):
    if len(sig) < N_SAMPLES:
        sig = np.pad(sig, (0, N_SAMPLES - len(sig)))
    return sig[:N_SAMPLES]

def to_mel(sig):
    m = librosa.feature.melspectrogram(y=sig, sr=SR, n_fft=N_FFT, hop_length=HOP, n_mels=N_MELS)
    db = librosa.power_to_db(m, ref=np.max)
    return ((db + 40) / 20).astype(np.float32)

@torch.no_grad()
def classify(path):
    """Return {class: probability} for an audio file of any length."""
    if path is None:
        return None
    sig, _ = librosa.load(path, sr=SR, mono=True)
    chunks = [sig[i:i + N_SAMPLES] for i in range(0, max(len(sig), 1), N_SAMPLES)]
    if len(chunks) > 1 and len(chunks[-1]) < SR:      # drop a tiny trailing chunk
        chunks = chunks[:-1]
    x = torch.from_numpy(np.stack([to_mel(fix_length(c)) for c in chunks]))[:, None]
    probs = model(x).softmax(1).mean(0)
    return {c: float(p) for c, p in zip(CLASSES, probs)}

## 5. Launch the web app
A sample clip from ESC-50 is downloaded so you can test the app with one click.

In [ ]:
import gradio as gr

sample = "dog_example.wav"
if not Path(sample).exists():
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/karolpiczak/ESC-50/master/audio/1-100032-A-0.wav", sample)

demo = gr.Interface(
    fn=classify,
    inputs=gr.Audio(sources=["upload", "microphone"], type="filepath", label="Audio (upload or record)"),
    outputs=gr.Label(num_top_classes=5, label="Top-5 predictions (confidence)"),
    examples=[sample],
    cache_examples=False,
    title="Environmental Sound Classifier (ESC-50)",
    description="ResNet18 fine-tuned on log-mel spectrograms. Recognizes 50 everyday sound classes "
                "(dog, rain, siren, keyboard typing, ...). Audio longer than 5 s is split into windows and averaged.",
)
demo.launch(share=True, debug=False)

**Note:** the public link works while this notebook is running (and for up to ~72 hours). The model only knows the 50 ESC-50 classes — a sound outside them is still assigned to the closest class, so check the confidence score.